In [ ]:
# 1) Install Gradio + mount Google Drive
%pip -q install gradio
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 2) Load the PyTorch model (ResNet50 + GPT-2) trained on Kaggle
import os, json, math
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import models as tvm
from transformers import GPT2TokenizerFast, GPT2LMHeadModel, GPT2Config
from PIL import Image

# >>> change this folder: it must contain best.pt and config.json (both from /kaggle/working/xray_report/)
MODEL_DIR = "/content/drive/MyDrive/xray_report"

best_path, cfg_path = os.path.join(MODEL_DIR, "best.pt"), os.path.join(MODEL_DIR, "config.json")
for p in (best_path, cfg_path):
    if not os.path.isfile(p):
        raise FileNotFoundError(f"File not found: {p}")

cfg = json.load(open(cfg_path))
ENCODER, IMG_SIZE, MAX_LEN = cfg["ENCODER"], int(cfg["IMG_SIZE"]), int(cfg["MAX_LEN"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")     # same original GPT-2 vocabulary used in training
tokenizer.pad_token = tokenizer.eos_token
BOS_ID = EOS_ID = PAD_ID = tokenizer.eos_token_id

IMG_MEAN, IMG_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class XrayReportModel(nn.Module):          # identical to the training notebook (minus augmentation)
    def __init__(self, lm, encoder_name, grid, p_drop=0.1):
        super().__init__()
        self.lm, self.grid, self.n_vis = lm, grid, 2 * grid * grid
        E = lm.config.n_embd
        net = getattr(tvm, encoder_name)(weights=None)     # weights come from best.pt, no download needed
        self.encoder = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool, net.layer1, net.layer2, net.layer3, net.layer4)
        self.proj = nn.Sequential(nn.Linear(net.fc.in_features, E), nn.GELU(), nn.Linear(E, E))
        self.pos  = nn.Parameter(torch.zeros(1, grid * grid, E))
        self.view = nn.Parameter(torch.zeros(1, 2, 1, E))
        self.norm = nn.LayerNorm(E)
        self.drop = nn.Dropout(p_drop)
        self.register_buffer("mean", torch.tensor(IMG_MEAN).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std",  torch.tensor(IMG_STD).view(1, 3, 1, 1), persistent=False)

    def encode_images(self, x_u8):
        B, _, H, W = x_u8.shape
        x = x_u8.reshape(B * 2, 1, H, W).float().div(255.0)
        x = (x.expand(-1, 3, -1, -1) - self.mean) / self.std
        f = F.adaptive_avg_pool2d(self.encoder(x), self.grid).flatten(2).transpose(1, 2)
        t = (self.proj(f) + self.pos).view(B, 2, self.grid * self.grid, -1) + self.view
        return self.drop(self.norm(t.reshape(B, self.n_vis, -1)))

    @torch.no_grad()
    def generate(self, imgs, max_new_tokens, num_beams=3, no_repeat_ngram_size=5):
        self.eval()
        prefix = self.encode_images(imgs)
        bos = self.lm.transformer.wte(torch.full((imgs.size(0), 1), BOS_ID, device=imgs.device))
        embeds = torch.cat([prefix.to(bos.dtype), bos], 1)
        attn = torch.ones(embeds.shape[:2], dtype=torch.long, device=embeds.device)
        return self.lm.generate(inputs_embeds=embeds, attention_mask=attn, max_new_tokens=max_new_tokens,
                                min_new_tokens=5, num_beams=num_beams, no_repeat_ngram_size=no_repeat_ngram_size,
                                early_stopping=True, do_sample=False, eos_token_id=EOS_ID, pad_token_id=PAD_ID,
                                use_cache=True)

lm = GPT2LMHeadModel(GPT2Config())          # default config = the original 124M GPT-2 architecture
model = XrayReportModel(lm, ENCODER, max(1, IMG_SIZE // 32))
state = torch.load(best_path, map_location="cpu")
missing, unexpected = model.load_state_dict(state, strict=False)
print("missing keys:", missing, "| unexpected keys:", unexpected)
model = model.to(DEVICE).eval()
print(f"Model loaded on {DEVICE} | encoder {ENCODER} | img {IMG_SIZE} | max_len {MAX_LEN}")


In [ ]:
# 3) Gradio UI
import gradio as gr

def prep(img):
    # same preprocessing as training: grayscale -> IMG_SIZE x IMG_SIZE bilinear -> uint8 (the model divides by 255 itself)
    arr = np.asarray(img.convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)
    return torch.from_numpy(arr)

def generate_report(frontal, lateral, num_beams):
    if frontal is None and lateral is None:
        raise gr.Error("Please upload at least one X-ray image.")
    note = ""
    if frontal is None or lateral is None:
        note = ("Note: only one view was uploaded, so it was used for both inputs. The model was trained on real "
                "frontal + lateral pairs, so the result is less reliable.\n\n")
        frontal = frontal if frontal is not None else lateral
        lateral = lateral if lateral is not None else frontal
    x = torch.stack([prep(frontal), prep(lateral)]).unsqueeze(0).to(DEVICE)     # (1, 2, H, W) uint8
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        out = model.generate(x, max_new_tokens=MAX_LEN - 1, num_beams=int(num_beams))
    row = out[0].tolist()
    if EOS_ID in row:
        row = row[:row.index(EOS_ID)]
    report = tokenizer.decode(row, skip_special_tokens=True).strip()
    return note + (report or "The model returned an empty report.")

demo = gr.Interface(
    fn=generate_report,
    inputs=[
        gr.Image(type="pil", label="Frontal X-ray (PA/AP)"),
        gr.Image(type="pil", label="Lateral X-ray"),
        gr.Slider(1, 5, value=3, step=1, label="Beam size"),
    ],
    outputs=gr.Textbox(label="Generated report", lines=12),
    title="Chest X-ray Report Generator (ResNet50 + GPT-2)",
    description="Upload the frontal and lateral chest X-rays. Research demo only - NOT for clinical use.",
    flagging_mode="never",
)
demo.launch(share=True, debug=True)
